In [ ]:
import MeshFEM
import py_newton_optimizer, benchmark, sim_utils
import mesh, viewer
import numpy as np
import matplotlib
import metric_fitting

In [ ]:
FIX_BOUNDARY = False

In [ ]:
m = mesh.Mesh('../3rdparty/MeshFEM/misc/examples/meshes/lilium.msh')

In [ ]:
# Run at higher resolution
import igl
m = mesh.Mesh(*igl.loop(m.vertices(), m.elements()))

In [ ]:
# Construct the metric fitting problem.
# Note: here the target metric is taken from the input surface.
# However, it is possible to overwrite the target metric either
# by applying a different immersion and calling
# `mf.programCurrentMetric()` or by manually calling
# `mf.setTargetMetric(ei, G)` for each element.
mf = metric_fitting.MetricFitter(m)

In [ ]:
# Generate a flattened initial configuration
# by aligning the boundary curve with the xy plane and then squashing
# nearly flat in in the z direction (slightly biasing towards the target shape)
import registration
bv = m.boundaryVertices()
R, t = registration.align_points_with_axes_xform(m.vertices()[bv])

squashedVertices = (m.vertices() + t) @ R
squashedVertices[:, 2] *= 0.005

mf.setVars(squashedVertices.ravel())

In [ ]:
view = viewer.Viewer(MeshFEM.EmbeddedMesh(m, mf))
view.show()

In [ ]:
view.setCameraParams(((1.7158632487838523, -3.4436707333058307, -1.9622883210997382), (-0.2621074609914764, 0.3797041381678899, -0.8872003417215216), (0.1588938840265548, 0.07134387298073316, 0.002046899267719321)))

In [ ]:
maxDistortion = np.max(np.sqrt(mf.metricDistSq()))
def metricErrorScalarField():
    return {'data': np.sqrt(mf.metricDistSq()), 'vmin': 0, 'vmax': maxDistortion, 'colormap': matplotlib.cm.coolwarm}
mf.setCustomIterationCallback(view.updater(updateFrequency=2, scalarFieldEvaluator=metricErrorScalarField))

In [ ]:
if FIX_BOUNDARY:
    boundaryVars = np.ravel([(3 * bvi, 3 * bvi + 1, 3 * bvi + 2) for bvi in bv])
    mf.setFixedVars(boundaryVars)
else:
    mf.hessianShift = 1e-8

In [ ]:
opt = mf.optimizer() # Create a solver
opt.options.verbose = False
opt.options.niter = 1000

In [ ]:
benchmark.reset()
mf.bendingStiffness = 1e-5
opt.optimize()
benchmark.report()

In [ ]:
mf.bendingStiffness = 1e-7
opt.optimize()
mf.bendingStiffness = 1e-9
opt.optimize();